# 02 — Data Quality
This notebook profiles the complete consumption, weather, and calendar datasets for the configured analysis period. It reports issues but does not silently alter the data.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
START_YEAR = 2021
END_YEAR = 2025

PROJECT_ROOT, PROCESSED_DIR

(WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk'),
 WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/data/processed'))

In [10]:
from src.ontario_peak_risk.data_quality.build_data_quality_report import build_data_quality_report

quality = build_data_quality_report(
    processed_directory=PROCESSED_DIR,
    project_root=PROJECT_ROOT,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

quality.keys()

Data Quality Report: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\docs\Data_Quality_Report.md
dataset_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_quality_dataset_summary.csv
column_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_quality_column_summary.csv
key_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_quality_key_summary.csv
temporal_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_quality_temporal_summary.csv
numeric_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\data_quality_numeric_summary.csv
category_summary: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports

dict_keys(['dataset_summary', 'column_summary', 'key_summary', 'temporal_summary', 'numeric_summary', 'category_summary', 'join_readiness'])

## Dataset overview

In [4]:
quality['dataset_summary']

,dataset,rows,columns,memory_mb,exact_duplicate_rows,minimum_timestamp,maximum_timestamp,missing_timestamp_rows,source_file_count
0,consumption,1046461,11,414.560,0,2021-01-01,2025-12-31 23:00:00,0,6
1,weather,87648,33,59.142,0,2021-01-01,2025-12-31 23:00:00,0,2
2,calendar,43824,45,32.081,0,2021-01-01,2025-12-31 23:00:00,0,1


## Key integrity

In [5]:
quality['key_summary']

,dataset,expected_key,available_key_columns,missing_key_rows,duplicate_key_rows,duplicate_key_groups,key_is_unique
0,consumption,FSA + TIMESTAMP + CUSTOMER_TYPE + PRICE_PLAN,FSA + TIMESTAMP + CUSTOMER_TYPE + PRICE_PLAN,0,0,0,True
1,weather,Climate ID + Date/Time (LST),Climate ID + Date/Time (LST),0,0,0,True
2,calendar,timestamp_local,timestamp_local,0,0,0,True


## Temporal completeness

In [6]:
quality['temporal_summary'].sort_values(['dataset', 'entity'])

,dataset,entity,minimum_timestamp,maximum_timestamp,observed_unique_hours,expected_continuous_hours,missing_hour_count,duplicate_timestamp_rows,days_with_less_than_24_unique_hours,days_with_more_than_24_unique_hours,minimum_hours_in_a_day,maximum_hours_in_a_day,first_missing_hour_examples
8,calendar,ALL,2021-01-01,2025-12-31 23:00:00,43824,43824,0,0,0,0,24,24,
0,consumption,FSA=L4T,2021-01-01,2025-12-31 23:00:00,43824,43824,0,215697,0,0,24,24,
1,consumption,FSA=M5R,2021-01-01,2025-12-31 23:00:00,43824,43824,0,101596,0,0,24,24,
2,consumption,FSA=M5S,2021-01-01,2025-12-31 23:00:00,43824,43824,0,103032,0,0,24,24,
3,consumption,FSA=M6G,2021-01-01,2025-12-31 23:00:00,43824,43824,0,203915,0,0,24,24,
4,consumption,FSA=M9R,2021-01-01,2025-12-31 23:00:00,43824,43824,0,209729,0,0,24,24,
5,consumption,FSA=M9W,2021-01-01,2025-12-31 23:00:00,43824,43824,0,212492,0,0,24,24,
6,weather,Climate ID=6158355 | Station Name=TORONTO CITY,2021-01-01,2025-12-31 23:00:00,43824,43824,0,0,0,0,24,24,
7,weather,Climate ID=6158731 | Station Name=TORONTO INTL A,2021-01-01,2025-12-31 23:00:00,43824,43824,0,0,0,0,24,24,


## Columns with the highest missingness

In [7]:
quality['column_summary'].sort_values(
    ['missing_pct', 'dataset'], ascending=[False, True]
).head(30)

,dataset,column,dtype,rows,non_null_count,missing_count,missing_pct,unique_count,constant_column
20,weather,Flag,float64,87648,0,87648,100.0000,0,True
22,weather,Temp Flag,float64,87648,0,87648,100.0000,0,True
28,weather,Precip. Amount Flag,float64,87648,0,87648,100.0000,0,True
30,weather,Wind Dir Flag,float64,87648,0,87648,100.0000,0,True
32,weather,Wind Spd Flag,float64,87648,0,87648,100.0000,0,True
36,weather,Stn Press Flag,float64,87648,0,87648,100.0000,0,True
38,weather,Hmdx Flag,float64,87648,0,87648,100.0000,0,True
40,weather,Wind Chill Flag,float64,87648,0,87648,100.0000,0,True
24,weather,Dew Point Temp Flag,object,87648,1,87647,99.9989,1,True
26,weather,Rel Hum Flag,object,87648,1,87647,99.9989,1,True


## Join readiness

In [8]:
quality['join_readiness']

,dataset,entity,unique_timestamps,timestamps_missing_from_calendar,calendar_timestamps_without_entity_data
0,consumption,FSA=L4T,43824,0,0
1,consumption,FSA=M5R,43824,0,0
2,consumption,FSA=M5S,43824,0,0
3,consumption,FSA=M6G,43824,0,0
4,consumption,FSA=M9R,43824,0,0
5,consumption,FSA=M9W,43824,0,0
6,weather,Climate ID=6158355 | Station=TORONTO CITY,43824,0,0
7,weather,Climate ID=6158731 | Station=TORONTO INTL A,43824,0,0


## Numeric variables and potential IQR outliers

In [9]:
quality['numeric_summary'].sort_values(
    'iqr_outlier_pct', ascending=False
).head(30)

,dataset,column,count,minimum,p01,p25,median,mean,p75,p99,maximum,standard_deviation,iqr_outlier_count,iqr_outlier_pct
21,weather,Visibility (km),43817,0.0,1.6,24.1,24.1,21.971529,24.1,24.1,80.5,5.448904,7311,16.6853
40,calendar,is_friday,43824,0.0,0.0,0.0,0.0,0.142935,0.0,1.0,1.0,0.350011,6264,14.2935
39,calendar,is_monday,43824,0.0,0.0,0.0,0.0,0.142935,0.0,1.0,1.0,0.350011,6264,14.2935
15,weather,Precip. Amount (mm),43783,0.0,0.0,0.0,0.0,0.090213,0.0,2.3,45.0,0.641122,3597,8.2155
1,consumption,TOTAL_CONSUMPTION,1046461,4.6,17.4,183.6,559.6,2209.34386,3584.4,12209.48,22747.4,3083.217289,56783,5.4262
52,calendar,is_long_weekend,43824,0.0,0.0,0.0,0.0,0.044359,0.0,1.0,1.0,0.205894,1944,4.4359
43,calendar,is_month_start,43824,0.0,0.0,0.0,0.0,0.032859,0.0,1.0,1.0,0.178269,1440,3.2859
44,calendar,is_month_end,43824,0.0,0.0,0.0,0.0,0.032859,0.0,1.0,1.0,0.178269,1440,3.2859
47,calendar,is_public_holiday,43824,0.0,0.0,0.0,0.0,0.026835,0.0,1.0,1.0,0.161602,1176,2.6835
48,calendar,is_day_before_holiday,43824,0.0,0.0,0.0,0.0,0.026835,0.0,1.0,1.0,0.161602,1176,2.6835


## Next action
Review `docs/Data_Quality_Report.md` and investigate issues before creating the Master Dataset. Outliers and missing values must not be removed solely because they were detected by this report.